# Chapter 9 — Polymorphism: When IS-A Meets HAS-A

---

# 9.0 Introduction — When Two Good Ideas Clash

So far, we have learned two powerful ideas:

---

## Idea 1 — Composition (HAS-A)

- A Library **HAS-A** collection of Books  
- A Bank **HAS-A** collection of Accounts  

---

## Idea 2 — Inheritance (IS-A)

- A Book **IS-A** LibraryItem  
- A DVD **IS-A** LibraryItem  
- A Magazine **IS-A** LibraryItem  

- A SavingsAccount **IS-A** Account  
- A MortgageAccount **IS-A** Account  

---

Individually, both ideas make sense.

Individually, both ideas improve our design.

---

## But Now We Try to Combine Them…

And this is where the problem appears.

---

👉 We try to use **HAS-A (composition)**  
👉 together with **IS-A (inheritance)**  

And suddenly…

👉 the design becomes clumsy again.

---

# 9.0.1 The Clash in the Library System

Let us look at our current design.

```mermaid
classDiagram

class LibraryItem {
    -title : String
    -available : boolean
    +display() void
}

class Book
class DVD
class Magazine

class Library {
    -books : ArrayList~Book~
    -dvds : ArrayList~DVD~
    -magazines : ArrayList~Magazine~
}

LibraryItem <|-- Book
LibraryItem <|-- DVD
LibraryItem <|-- Magazine

Library "1" --> "*" Book
Library "1" --> "*" DVD
Library "1" --> "*" Magazine
```

---

## What We Have Done

We used inheritance correctly:

- Book **IS-A** LibraryItem  
- DVD **IS-A** LibraryItem  
- Magazine **IS-A** LibraryItem  

---

But now look at composition:

- Library **HAS-A** books  
- Library **HAS-A** DVDs  
- Library **HAS-A** magazines  

---

## The Problem

Even though everything is a `LibraryItem`…

👉 the `Library` is still treating them as separate types  

---

So we end up with:

- three lists ❌  
- three add methods ❌  
- repeated logic ❌  

---

👉 This is the **clunkiness**

---

# 9.0.2 The Same Clash in the Bank System

Now look at the bank:

```mermaid
classDiagram

class Account {
    -name : String
    -balance : double
    +getBalance() double
}

class SavingsAccount
class MortgageAccount

class Bank {
    -savingsAccounts : ArrayList~SavingsAccount~
    -mortgageAccounts : ArrayList~MortgageAccount~
}

Account <|-- SavingsAccount
Account <|-- MortgageAccount

Bank "1" --> "*" SavingsAccount
Bank "1" --> "*" MortgageAccount
```

---

Again:

Inheritance says:

- SavingsAccount **IS-A** Account  
- MortgageAccount **IS-A** Account  

---

But composition says:

- Bank **HAS-A** savings accounts  
- Bank **HAS-A** mortgage accounts  

---

## The Same Problem

Even though both are `Account` objects…

👉 the `Bank` still splits them into separate collections  

---

So again we get:

- multiple lists ❌  
- multiple methods ❌  
- unnecessary complexity ❌  

---

# 9.0.3 What We *Want* Instead

What we really want is this:

---

## Cleaner Library Design

```mermaid
classDiagram

class LibraryItem
class Book
class DVD
class Magazine

class Library {
    -items : ArrayList~LibraryItem~
}

LibraryItem <|-- Book
LibraryItem <|-- DVD
LibraryItem <|-- Magazine

Library "1" --> "*" LibraryItem
```

---

## Cleaner Bank Design

```mermaid
classDiagram

class Account
class SavingsAccount
class MortgageAccount

class Bank {
    -accounts : ArrayList~Account~
}

Account <|-- SavingsAccount
Account <|-- MortgageAccount

Bank "1" --> "*" Account
```

---

## What Changed?

We finally made composition match inheritance.

---

Instead of:

👉 many collections of child types  

we now have:

👉 one collection of the parent type  

---

## Why This Feels Better

- simpler design  
- fewer variables  
- fewer methods  
- easier to extend  

---

But now we have a deeper question…

---

# 9.0.4 The Big Question

If everything is stored in an Arraylist of parent type:

- `LibraryItem`
- `Account`

---

👉 How does Java know what the *actual* object is?  

👉 How does it know which `display()` to run?  

👉 How does one list hold different types safely?  

---

# 9.0.5 The Missing Piece

To make this design work, we need one more idea.

---

👉 **Polymorphism**

---

## First Intuition

Polymorphism means:

👉 one type  
👉 many possible forms  

---

For example:

- a `LibraryItem` reference might point to a `Book`  
- or a `DVD`  
- or a `Magazine`  

---

And:

- an `Account` reference might point to a `SavingsAccount`  
- or a `MortgageAccount`  

---

## Key Insight

Polymorphism is what allows:

👉 IS-A (inheritance)  
👉 to work cleanly with HAS-A (composition)  

---

Without it…

👉 our designs stay clunky  

With it…

👉 everything becomes simple again  

---

We are now ready to see how this works in Java.

## 9.1 Making It Real — First Rebuild the Inheritance Structure

Before we can use polymorphism, we must first rebuild the inheritance structure from the last chapter.

We need:

- a parent class: `LibraryItem`
- three child classes:
  - `Book`
  - `DVD`
  - `Magazine`

---

## 9.1.1 Class Diagram — The Inheritance Structure

Look carefully at this diagram first.

This part is only about **IS-A**.

We are not yet building the `Library` class.

We are only rebuilding the inheritance hierarchy that we need before polymorphism can work.

```mermaid
classDiagram

class LibraryItem {
    -title : String
    -available : boolean
    +LibraryItem(title)
    +display() void
    +checkOut() void
    +returnItem() void
}

class Book {
    -author : String
    -isbn : String
    +Book(title, author, isbn)
    +display() void
}

class DVD {
    -director : String
    -releaseYear : int
    +DVD(title, director, releaseYear)
    +display() void
}

class Magazine {
    -issueNumber : int
    +Magazine(title, issueNumber)
    +display() void
}

LibraryItem <|-- Book
LibraryItem <|-- DVD
LibraryItem <|-- Magazine
```

---

Now let us implement that diagram in Java.

In [ ]:
class LibraryItem {

    private String title;
    private boolean available;

    public LibraryItem(String title) {
        this.title = title;
        this.available = true;
    }

    public void display() {
        System.out.println("Title: " + title);
        System.out.println("Available: " + available);
    }

    public void checkOut() {
        available = false;
    }

    public void returnItem() {
        available = true;
    }
}

class Book extends LibraryItem {

    private String author;
    private String isbn;

    public Book(String title, String author, String isbn) {
        super(title);
        this.author = author;
        this.isbn = isbn;
    }

    public void display() {
        super.display();
        System.out.println("Author: " + author);
        System.out.println("ISBN: " + isbn);
    }
}

class DVD extends LibraryItem {

    private String director;
    private int releaseYear;

    public DVD(String title, String director, int releaseYear) {
        super(title);
        this.director = director;
        this.releaseYear = releaseYear;
    }

    public void display() {
        super.display();
        System.out.println("Director: " + director);
        System.out.println("Release year: " + releaseYear);
    }
}

class Magazine extends LibraryItem {

    private int issueNumber;

    public Magazine(String title, int issueNumber) {
        super(title);
        this.issueNumber = issueNumber;
    }

    public void display() {
        super.display();
        System.out.println("Issue number: " + issueNumber);
    }
}

---

## 9.1.2 Quick Check — Does the Inheritance Structure Work?

Before moving on to polymorphism, let us quickly test that these classes work properly.

We will make one object of each type and call `display()`.

This is still just inheritance and overriding.

We have not yet used the new idea of polymorphism.

In [ ]:
//Lets make some objects
Book b1 = new Book("The Hobbit", "J.R.R. Tolkien", "123");
DVD d1 = new DVD("Inception", "Christopher Nolan", 2010);
Magazine m1 = new Magazine("Science Weekly", 42);

b1.display();
d1.display();
m1.display();

So far so good!

So the inheritance structure is working.

Each child class:

- inherits from `LibraryItem`
- has its own constructor
- has its own overridden `display()` method

---

## 9.1.3 Now We Introduce the New Idea

Up to now, we have used the child classes directly:

- `Book`
- `DVD`
- `Magazine`

But now we ask a new question.

Since all three classes are types of `LibraryItem`...

can we use the **parent type** directly?

---

## 9.1.4 Parent Type References

Look at the next code very carefully.

The type on the left is always the same:

`LibraryItem`

But the actual object created on the right changes.

In [ ]:
// Make objects again - using parent type!!!
LibraryItem item1 = new Book("The Hobbit", "J.R.R. Tolkien", "123");
LibraryItem item2 = new DVD("Inception", "Christopher Nolan", 2010);
LibraryItem item3 = new Magazine("Science Weekly", 42);

This is legal because:

- a `Book` **IS-A** `LibraryItem`
- a `DVD` **IS-A** `LibraryItem`
- a `Magazine` **IS-A** `LibraryItem`

So a variable of type `LibraryItem` is allowed to refer to any of those child objects.

---

## 9.1.5 What Happens If We Call `display()`?

Now comes the key test.

All three variables are declared as `LibraryItem`.

So what happens here?

In [ ]:
// Different displays???? Wow!!!!
item1.display();
item2.display();
item3.display();

---

## What Just Happened?

Even though all three variables have the same declared type:

`LibraryItem`

Java still ran the correct version of `display()` for each object.

- `item1` refers to a `Book` → so `Book.display()` runs  
- `item2` refers to a `DVD` → so `DVD.display()` runs  
- `item3` refers to a `Magazine` → so `Magazine.display()` runs  

---

## Key Idea

The reference type is the same - thast teh parent type.

But the actual object type is different becuae we used the child constructros.

Java uses the **actual object** to decide which overridden method to run.

👉 That is polymorphism.

---

## 9.1.6 Exercise

Make three more parent-type references below:

- one `LibraryItem` referring to a new `Book`
- one `LibraryItem` referring to a new `DVD`
- one `LibraryItem` referring to a new `Magazine`

Then call `display()` on all three.

In [ ]:
// Exercise
// Create three more variables of type LibraryItem below

// LibraryItem item4 = ...
// LibraryItem item5 = ...
// LibraryItem item6 = ...

// Now call display() on them

---

## 9.1.7 Why This Matters

This is the missing step we needed.

In the previous section, the cleaner class diagrams used:

- one collection of `LibraryItem`

But that design only works if Java allows one parent type to refer to many different child objects.

Now we have seen that it does.

So we are ready for the next step:

👉 storing many different child objects together in one collection of the parent type

---

## 9.2 Bringing It All Together — One Collection Using Polymorphism

We are now ready to combine everything:

- inheritance (**IS-A**)  
- composition (**HAS-A**)  
- and polymorphism  

---

## 9.2.1 Full Class Diagram — Clean Library Design

Look carefully at this diagram.

This is the **complete design** we are now aiming to implement.

```mermaid
classDiagram

class LibraryItem {
    -title : String
    -available : boolean
    +LibraryItem(title)
    +display() void
    +checkOut() void
    +returnItem() void
}

class Book {
    -author : String
    -isbn : String
    +Book(title, author, isbn)
    +display() void
}

class DVD {
    -director : String
    -releaseYear : int
    +DVD(title, director, releaseYear)
    +display() void
}

class Magazine {
    -issueNumber : int
    +Magazine(title, issueNumber)
    +display() void
}

class Library {
    -items : ArrayList~LibraryItem~
    +addItem(item : LibraryItem) void
    +displayAll() void
}

LibraryItem <|-- Book
LibraryItem <|-- DVD
LibraryItem <|-- Magazine

Library "1" --> "*" LibraryItem
```

---

## What This Diagram Shows

### Inheritance (IS-A)

- Book **IS-A** LibraryItem  
- DVD **IS-A** LibraryItem  
- Magazine **IS-A** LibraryItem  

---

### Composition (HAS-A)

- Library **HAS-A** collection of LibraryItem objects  

---

### The Key Design Improvement

Instead of:

- three lists ❌  
- three add methods ❌  

we now have:

- one list ✅  
- one add method ✅  

---

## Why This Works

Because:

👉 a `Book` **IS-A** `LibraryItem`  
👉 a `DVD` **IS-A** `LibraryItem`  
👉 a `Magazine` **IS-A** `LibraryItem`  

---

So one collection of `LibraryItem` can store all of them.

---

## 9.2.2 Implementing the Library Class

We have already written:

- `LibraryItem`
- `Book`
- `DVD`
- `Magazine`

Now we add the new class from the diagram:

👉 `Library`

In [ ]:
// Lets make the libray - it will just hold one ArrayList - NOT THREE
import java.util.ArrayList;

class Library {

    private ArrayList<LibraryItem> items;

    public Library() {
        items = new ArrayList<>();
    }

    public void addItem(LibraryItem item) {
        items.add(item);
        //👉 we only use ONE method: `addItem(...)`  
        //👉 we do NOT care what specific type the object is 
    }

    public void displayAll() { 
        for (LibraryItem item : items) { //Note how similar this is to python code yuo learned last year
            item.display();
            System.out.println();
        }
    }
}

---

## 9.2.3 Testing the Full System

Now we will create a `Library` and add different types of objects to it.

Notice:

👉 we only use ONE method: `addItem(...)`  
👉 we do NOT care what specific type the object is  

---

In [ ]:
// Lets make a library lobject and add some different child objects!
Library lib = new Library();

lib.addItem(new Book("The Hobbit", "J.R.R. Tolkien", "123"));
lib.addItem(new DVD("Inception", "Christopher Nolan", 2010));
lib.addItem(new Magazine("Science Weekly", 42));

---

## 9.2.4 Display Everything

Now we call one method:

👉 `displayAll()`

In [ ]:
lib.displayAll();

---

## Think Carefully About What Just Happened

Inside `displayAll()` we wrote:

```java
for (LibraryItem item : items) {
    item.display();
}
```

---

But each object in the list is actually different:

- Book  
- DVD  
- Magazine  

---

And yet:

👉 the correct `display()` method runs every time  

---

## This is the Key Moment

We have successfully combined:

- IS-A (inheritance)  
- HAS-A (composition)  
- polymorphism  

---

## Before vs After

### Before (Clunky Design)

- multiple lists  
- multiple add methods  
- multiple loops  

---

### After (Clean Design)

- one list  
- one add method  
- one loop  

---

## Final Key Idea

Polymorphism allows us to:

👉 treat different objects as the same type  
👉 while still preserving their individual behaviour  

---

This is what makes object-oriented design powerful.

---

## Exercise

1. Add two more items to the library:
   - another `Book`
   - another `DVD`

2. Call `displayAll()` again

3. Verify:
   - the correct `display()` method runs for every object


In [ ]:
// Exercise

// lib.addItem(...);
// lib.addItem(...);

// lib.displayAll();






## 9.3 Practice Example — Bank System (Polymorphism + Rules)

We now repeat the same idea…

👉 but in a different system  

This helps confirm that polymorphism is not just for libraries — it works everywhere.

---

## 9.3.1 Full Class Diagram — Clean Bank Design

Study this diagram carefully before we write any code.

```mermaid
classDiagram

class Account {
    -name : String
    -balance : double
    +Account(name, balance)
    +getBalance() double
    +display() void
}

class SavingsAccount {
    +SavingsAccount(name, balance)
}

class MortgageAccount {
    +MortgageAccount(name, balance)
}

class Bank {
    -accounts : ArrayList~Account~
    +Bank()
    +addAccount(acc : Account) void
    +displayAll() void
}

Account <|-- SavingsAccount
Account <|-- MortgageAccount

Bank "1" --> "*" Account
```

---

## What This Shows

### Inheritance (IS-A)

- SavingsAccount **IS-A** Account  
- MortgageAccount **IS-A** Account  

---

### Composition (HAS-A)

- Bank **HAS-A** collection of Accounts  

---

### Key Idea

👉 One list of `Account` objects  
👉 Not separate lists for each type  

---

## 9.3.2 Step 1 — Build the Parent Class

We start with the `Account` class from the diagram.

---

### Exercise

Write the `Account` class with:

- private `name`
- private `balance`
- constructor
- `getBalance()`
- `display()`

```mermaid
classDiagram

class Account {
    -name : String
    -balance : double
    +Account(name, balance)
    +getBalance() double
    +display() void
}

```

In [ ]:
class Account {

//Exercise
    
}

---

## 9.3.3 Step 2 — Create Child Classes

Now we implement:

- `SavingsAccount`
- `MortgageAccount`

Both should:

👉 extend `Account`  
👉 call `super(...)` in their constructors  

---

```mermaid
classDiagram

class Account {
    -name : String
    -balance : double
    +Account(name, balance)
    +getBalance() double
    +display() void
}

class SavingsAccount {
    +SavingsAccount(name, balance)
}

class MortgageAccount {
    +MortgageAccount(name, balance)
}

Account <|-- SavingsAccount
Account <|-- MortgageAccount

```

### Exercise

Complete the two classes below such that we have rules:

- Savings accounts should NOT have negative balances  
- Mortgage accounts should NOT have positive balances  

---

Modify the constructors:

- If a SavingsAccount is given a negative balance → set it to 0  
- If a MortgageAccount is given a positive balance → set it to 0  

In [ ]:
// Complete the classes
class SavingsAccount extends Account {

    public SavingsAccount(String name, double balance) {
        super(name, balance);
    }
}

class MortgageAccount extends Account {

    public MortgageAccount(String name, double balance) {
        super(name, balance);
    }
}

---

## 9.3.4 Quick Test

Let us test the rules before moving on.

In [ ]:
SavingsAccount s1 = new SavingsAccount("Alice", 100);
SavingsAccount s2 = new SavingsAccount("Bob", -50);

MortgageAccount m1 = new MortgageAccount("Charlie", -200000);
MortgageAccount m2 = new MortgageAccount("Dana", 50000);

s1.display();
s2.display(); // should be 0
m1.display();
m2.display(); // should be 0

---

## 9.3.5 Step 4 — Build the Bank Class

Now we implement the composition part:

👉 `Bank HAS-A collection of Account`

---

```mermaid
classDiagram

class Bank {
    -accounts : ArrayList~Account~
    +Bank()
    +addAccount(acc : Account) void
    +displayAll() void
}

```

### Exercise

Write the `Bank` class with:

- `ArrayList<Account>`
- constructor
- `addAccount(...)`
- `displayAll()`

In [ ]:
import java.util.ArrayList;

// Exercise - add the code
class Bank {

    // Look up at the Library example for inspiraton
}

---

## 9.3.6 Step 5 — Use Polymorphism

Now we bring everything together.

Notice:

👉 we only use ONE list  
👉 we only use ONE method  
👉 we mix different object types freely  

---

In [ ]:
Bank bank = new Bank();

bank.addAccount(new SavingsAccount("Alice", 100));
bank.addAccount(new SavingsAccount("Bob", -50));

bank.addAccount(new MortgageAccount("Charlie", -200000));
bank.addAccount(new MortgageAccount("Dana", 50000));

bank.displayAll();

---

## Final Insight

We have now repeated the same pattern:

- Library system ✅  
- Bank system ✅  

---

## The Pattern

1. Create a parent class  
2. Create child classes using `extends`  
3. Store everything using the parent type  
4. Use one loop to process all objects  

---

## Exercise

1. Add two more accounts:
   - one SavingsAccount
   - one MortgageAccount  

2. Display all accounts again  

3. Confirm:
   - rules still apply  
   - correct output still appears  

---

In [ ]:
// Exercise

// bank.addAccount(...);
// bank.addAccount(...);

// bank.displayAll();